In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.autograd import grad
from torch.optim import Adam, LBFGS

# Check for GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

# Define constants
REYNOLDS_NUMBER = 100
U_LID = 1.0
DOMAIN_SIZE = 1.0  # L in Re = UL/nu
NU = (U_LID * DOMAIN_SIZE) / REYNOLDS_NUMBER # Kinematic viscosity
RHO = 1.0 # Density (assumed 1 for simplicity)

# 1. Define the PINN Model
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        # 5 hidden layers, 50 neurons each
        self.net = nn.Sequential(
            nn.Linear(2, 50), # Input (x, y)
            nn.Tanh(), # Or nn.SiLU() - Tanh is common for PINNs due to smooth derivatives
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 50),
            nn.Tanh(),
            nn.Linear(50, 3) # Output (u, v, p)
        )

    def forward(self, x, y):
        # Ensure x and y are column vectors for concatenation
        inputs = torch.cat((x.view(-1, 1), y.view(-1, 1)), dim=1)
        outputs = self.net(inputs)
        u = outputs[:, 0].view(-1, 1)
        v = outputs[:, 1].view(-1, 1)
        p = outputs[:, 2].view(-1, 1)
        return u, v, p

# 2. Helper function to compute derivatives
def compute_derivatives(u, v, p, x, y):
    u_x = grad(u, x, torch.ones_like(u), create_graph=True, allow_unused=True)[0]
    u_y = grad(u, y, torch.ones_like(u), create_graph=True, allow_unused=True)[0]
    v_x = grad(v, x, torch.ones_like(v), create_graph=True, allow_unused=True)[0]
    v_y = grad(v, y, torch.ones_like(v), create_graph=True, allow_unused=True)[0]
    p_x = grad(p, x, torch.ones_like(p), create_graph=True, allow_unused=True)[0]
    p_y = grad(p, y, torch.ones_like(p), create_graph=True, allow_unused=True)[0]

    u_x = u_x if u_x is not None else torch.zeros_like(u)
    u_y = u_y if u_y is not None else torch.zeros_like(u)
    v_x = v_x if v_x is not None else torch.zeros_like(v)
    v_y = v_y if v_y is not None else torch.zeros_like(v)
    p_x = p_x if p_x is not None else torch.zeros_like(p)
    p_y = p_y if p_y is not None else torch.zeros_like(p)

    u_xx = grad(u_x, x, torch.ones_like(u_x), create_graph=True, allow_unused=True)[0]
    u_yy = grad(u_y, y, torch.ones_like(u_y), create_graph=True, allow_unused=True)[0]
    v_xx = grad(v_x, x, torch.ones_like(v_x), create_graph=True, allow_unused=True)[0]
    v_yy = grad(v_y, y, torch.ones_like(v_y), create_graph=True, allow_unused=True)[0]

    u_xx = u_xx if u_xx is not None else torch.zeros_like(u_x)
    u_yy = u_yy if u_yy is not None else torch.zeros_like(u_y)
    v_xx = v_xx if v_xx is not None else torch.zeros_like(v_x)
    v_yy = v_yy if v_yy is not None else torch.zeros_like(v_y)

    return u_x, u_y, v_x, v_y, p_x, p_y, u_xx, u_yy, v_xx, v_yy

# 3. Define the loss functions
def pde_loss(u, v, p, x, y):
    u_x, u_y, v_x, v_y, p_x, p_y, u_xx, u_yy, v_xx, v_yy = compute_derivatives(u, v, p, x, y)

    f_continuity = u_x + v_y
    f_momentum_x = u * u_x + v * u_y + p_x/RHO - NU * (u_xx + u_yy)
    f_momentum_y = u * v_x + v * v_y + p_y/RHO - NU * (v_xx + v_yy)

    loss_continuity = torch.mean(f_continuity**2)
    loss_momentum_x = torch.mean(f_momentum_x**2)
    loss_momentum_y = torch.mean(f_momentum_y**2)

    return loss_continuity + loss_momentum_x + loss_momentum_y

def boundary_loss(model, num_points_boundary, U_LID_val):
    x_top = torch.rand(num_points_boundary // 4, 1, requires_grad=True).to(device)
    y_top = torch.ones_like(x_top).to(device)
    u_top, v_top, _ = model(x_top, y_top)
    loss_top = torch.mean((u_top - U_LID_val)**2) + torch.mean(v_top**2)

    x_bottom = torch.rand(num_points_boundary // 4, 1, requires_grad=True).to(device)
    y_bottom = torch.zeros_like(x_bottom).to(device)
    u_bottom, v_bottom, _ = model(x_bottom, y_bottom)
    loss_bottom = torch.mean(u_bottom**2) + torch.mean(v_bottom**2)

    y_left = torch.rand(num_points_boundary // 4, 1, requires_grad=True).to(device)
    x_left = torch.zeros_like(y_left).to(device)
    u_left, v_left, _ = model(x_left, y_left)
    loss_left = torch.mean(u_left**2) + torch.mean(v_left**2)

    y_right = torch.rand(num_points_boundary // 4, 1, requires_grad=True).to(device)
    x_right = torch.ones_like(y_right).to(device)
    u_right, v_right, _ = model(x_right, y_right)
    loss_right = torch.mean(u_right**2) + torch.mean(v_right**2)

    return loss_top + loss_bottom + loss_left + loss_right

# 4. Modified Training Loop
def train_pinn(model, num_adam_epochs, num_lbfgs_epochs, num_collocation_points, num_boundary_points, adam_lr=1e-3):
    # Store tuples of (epoch_number, loss_value)
    history = []

    # --- Adam Optimizer Phase ---
    print("\n--- Starting Adam Optimization ---")
    optimizer_adam = Adam(model.parameters(), lr=adam_lr)
    for epoch in range(num_adam_epochs):
        optimizer_adam.zero_grad()

        x_collocation_adam = torch.rand(num_collocation_points, 1, requires_grad=True).to(device)
        y_collocation_adam = torch.rand(num_collocation_points, 1, requires_grad=True).to(device)
        u_pred_collocation, v_pred_collocation, p_pred_collocation = model(x_collocation_adam, y_collocation_adam)
        loss_pde = pde_loss(u_pred_collocation, v_pred_collocation, p_pred_collocation, x_collocation_adam, y_collocation_adam)

        loss_bc = boundary_loss(model, num_boundary_points, U_LID)
        loss = 100*loss_pde + loss_bc

        loss.backward()
        optimizer_adam.step()

        if (epoch + 1) % 100 == 0:
            print(f"Adam Epoch {epoch+1}/{num_adam_epochs}, Loss: {loss.item():.6f}, PDE Loss: {loss_pde.item():.6f}, BC Loss: {loss_bc.item():.6f}")
            history.append((epoch + 1, loss.item())) # Store (epoch, loss)

    # --- L-BFGS Optimizer Phase ---
    print("\n--- Starting L-BFGS Optimization ---")
    x_collocation_lbfgs = torch.rand(num_collocation_points, 1, requires_grad=True).to(device)
    y_collocation_lbfgs = torch.rand(num_collocation_points, 1, requires_grad=True).to(device)

    x_boundary_lbfgs_top = torch.rand(num_boundary_points // 4, 1, requires_grad=True).to(device)
    y_boundary_lbfgs_top = torch.ones_like(x_boundary_lbfgs_top).to(device)
    x_boundary_lbfgs_bottom = torch.rand(num_boundary_points // 4, 1, requires_grad=True).to(device)
    y_boundary_lbfgs_bottom = torch.zeros_like(x_boundary_lbfgs_bottom).to(device)
    y_boundary_lbfgs_left = torch.rand(num_boundary_points // 4, 1, requires_grad=True).to(device)
    x_boundary_lbfgs_left = torch.zeros_like(y_boundary_lbfgs_left).to(device)
    y_boundary_lbfgs_right = torch.rand(num_boundary_points // 4, 1, requires_grad=True).to(device)
    x_boundary_lbfgs_right = torch.ones_like(y_boundary_lbfgs_right).to(device)

    optimizer_lbfgs = LBFGS(model.parameters(),
                            lr=0.5,
                            max_iter=50,
                            history_size=100,
                            line_search_fn="strong_wolfe")

    # Keep track of the current 'global' epoch number for L-BFGS plotting
    current_global_epoch = num_adam_epochs

    def closure():
        optimizer_lbfgs.zero_grad()

        u_pred_collocation, v_pred_collocation, p_pred_collocation = model(x_collocation_lbfgs, y_collocation_lbfgs)
        loss_pde = pde_loss(u_pred_collocation, v_pred_collocation, p_pred_collocation, x_collocation_lbfgs, y_collocation_lbfgs)

        u_top, v_top, _ = model(x_boundary_lbfgs_top, y_boundary_lbfgs_top)
        loss_top = torch.mean((u_top - U_LID)**2) + torch.mean(v_top**2)
        u_bottom, v_bottom, _ = model(x_boundary_lbfgs_bottom, y_boundary_lbfgs_bottom)
        loss_bottom = torch.mean(u_bottom**2) + torch.mean(v_bottom**2)
        u_left, v_left, _ = model(x_boundary_lbfgs_left, y_boundary_lbfgs_left)
        loss_left = torch.mean(u_left**2) + torch.mean(v_left**2)
        u_right, v_right, _ = model(x_boundary_lbfgs_right, y_boundary_lbfgs_right)
        loss_right = torch.mean(u_right**2) + torch.mean(v_right**2)
        loss_bc = loss_top + loss_bottom + loss_left + loss_right

        loss = 1*loss_pde + loss_bc

        if loss.requires_grad:
            loss.backward()

        return loss

    for epoch_lbfgs in range(num_lbfgs_epochs):
        current_global_epoch += 1 # Increment global epoch for L-BFGS visualization
        loss_lbfgs = optimizer_lbfgs.step(closure)
        if (epoch_lbfgs + 1) % 10 == 0 or epoch_lbfgs == 0:
            print(f"L-BFGS Epoch {epoch_lbfgs+1}/{num_lbfgs_epochs} (Global: {current_global_epoch}), Loss: {loss_lbfgs.item():.6f}")
            history.append((current_global_epoch, loss_lbfgs.item())) # Store (global epoch, loss)

    return history

# Main execution
if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    model = PINN().to(device)

    NUM_ADAM_EPOCHS = 1000
    NUM_LBFGS_EPOCHS = 500
    NUM_COLLOCATION_POINTS = 10000
    NUM_BOUNDARY_POINTS = 10000
    ADAM_LEARNING_RATE = 1e-3

    print(f"Starting PINN training for Re={REYNOLDS_NUMBER} with {NUM_ADAM_EPOCHS} Adam epochs then {NUM_LBFGS_EPOCHS} L-BFGS epochs...")
    training_history_raw = train_pinn(model, NUM_ADAM_EPOCHS, NUM_LBFGS_EPOCHS,
                                      NUM_COLLOCATION_POINTS, NUM_BOUNDARY_POINTS,
                                      adam_lr=ADAM_LEARNING_RATE)
    print("Training finished.")

    # Unpack the history into separate epoch numbers and loss values for plotting
    epochs_for_plot = [item[0] for item in training_history_raw]
    losses_for_plot = [item[1] for item in training_history_raw]

    # --- Data Extraction for Plotting and Saving ---
    num_test_points = 100
    x_test_coords = torch.linspace(0, DOMAIN_SIZE, num_test_points).view(-1, 1)
    y_test_coords = torch.linspace(0, DOMAIN_SIZE, num_test_points).view(-1, 1)
    X_grid, Y_grid = torch.meshgrid(x_test_coords.squeeze(), y_test_coords.squeeze(), indexing='xy')

    X_flat_tensor = X_grid.flatten().unsqueeze(1).to(device)
    Y_flat_tensor = Y_grid.flatten().unsqueeze(1).to(device)

    model.eval()
    with torch.no_grad():
        u_pred_flat, v_pred_flat, p_pred_flat = model(X_flat_tensor, Y_flat_tensor)

    U_plot = u_pred_flat.cpu().numpy().reshape(num_test_points, num_test_points)
    V_plot = v_pred_flat.cpu().numpy().reshape(num_test_points, num_test_points)
    P_plot = p_pred_flat.cpu().numpy().reshape(num_test_points, num_test_points)

    # --- SAVE DATA TO .DAT FILE ---
    X_np_flat = X_flat_tensor.cpu().numpy()
    Y_np_flat = Y_flat_tensor.cpu().numpy()
    U_np_flat = u_pred_flat.cpu().numpy()
    V_np_flat = v_pred_flat.cpu().numpy()
    P_np_flat = p_pred_flat.cpu().numpy()

    output_data = np.hstack((X_np_flat, Y_np_flat, U_np_flat, V_np_flat, P_np_flat))
    output_filename = f"lid_driven_cavity_re{REYNOLDS_NUMBER}_pinn_results.dat"
    np.savetxt(output_filename, output_data, fmt='%.8e', delimiter='\t', header='X\tY\tU\tV\tP', comments='')
    print(f"Results saved to {output_filename}")


    # --- Plotting 2D contours ---
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    im1 = axs[0].imshow(U_plot.T, cmap='jet', origin='lower',
                             extent=[X_grid.min(), X_grid.max(), Y_grid.min(), Y_grid.max()])
    axs[0].set_title('U Velocity')
    axs[0].set_xlabel('X')
    axs[0].set_ylabel('Y')
    fig.colorbar(im1, ax=axs[0])

    im2 = axs[1].imshow(V_plot.T, cmap='jet', origin='lower',
                             extent=[X_grid.min(), X_grid.max(), Y_grid.min(), Y_grid.max()])
    axs[1].set_title('V Velocity')
    axs[1].set_xlabel('X')
    axs[1].set_ylabel('Y')
    fig.colorbar(im2, ax=axs[1])

    im3 = axs[2].imshow(P_plot.T, cmap='jet', origin='lower',
                             extent=[X_grid.min(), X_grid.max(), Y_grid.min(), Y_grid.max()])
    axs[2].set_title('Pressure')
    axs[2].set_xlabel('X')
    axs[2].set_ylabel('Y')
    fig.colorbar(im3, ax=axs[2])

    skip_quiver = 8
    axs[0].quiver(X_grid.cpu().numpy()[::skip_quiver, ::skip_quiver],
                     Y_grid.cpu().numpy()[::skip_quiver, ::skip_quiver],
                     U_plot[::skip_quiver, ::skip_quiver],
                     V_plot[::skip_quiver, ::skip_quiver], color='white', scale=5)
    axs[1].quiver(X_grid.cpu().numpy()[::skip_quiver, ::skip_quiver],
                     Y_grid.cpu().numpy()[::skip_quiver, ::skip_quiver],
                     U_plot[::skip_quiver, ::skip_quiver],
                     V_plot[::skip_quiver, ::skip_quiver], color='white', scale=5)

    plt.tight_layout()
    plt.show()

    # --- Plotting the loss history (corrected) ---
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_for_plot, losses_for_plot, label='Total Loss')
    plt.title('PINN Training Loss History (Adam then L-BFGS)')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.legend()
    plt.show()


    # --- Centerline Velocity Comparison with Ghia et al. ---
    ghia_u_y = np.array([1.000000000E+000, 9.765999913E-001, 9.688000083E-001, 9.609000087E-001, 9.531000257E-001, 8.515999913E-001, 7.343999743E-001, 6.172000170E-001, 5.000000000E-001, 4.530999959E-001, 2.813000083E-001, 1.719000041E-001, 1.015999988E-001, 7.029999793E-002, 6.250000000E-002, 5.469999835E-002, 0.000000000E+000])
    ghia_u_val = np.array([1.000000000E+000, 8.412299752E-001, 7.887099981E-001, 7.372199893E-001, 6.871700287E-001, 2.315099984E-001, 3.319999902E-003, -1.364099979E-001, -2.058099955E-001, -2.108999938E-001, -1.566199958E-001, -1.014999971E-001, -6.434000283E-002, -4.774999991E-002, -4.191999882E-002, -3.717000037E-002, 0.000000000E+000])

    ghia_v_x = np.array([0.963023302, 0.951718009, 0.945102686, 0.93853673, 0.900473934, 0.859843997, 0.801490916, 0.487460506, 0.222501975, 0.201174961, 0.142377567, 0.077359795, 0.061413902, 0.05450237, 0.04778831])
    ghia_v_val = np.array([-0.059722222, -0.075, -0.088888889, -0.104166667, -0.166666667, -0.223611111, -0.248611111, 0.052777778, 0.173611111, 0.173611111, 0.161111111, 0.123611111, 0.105555556, 0.1, 0.088888889])

    # Extract centerline data from PINN predictions
    x_half_idx = np.argmin(np.abs(X_grid.cpu().numpy()[0, :] - 0.5))
    u_centerline_pinn = U_plot[:, x_half_idx]

    y_half_idx = np.argmin(np.abs(Y_grid.cpu().numpy()[:, 0] - 0.5))
    v_centerline_pinn = V_plot[y_half_idx, :]

    # Plotting Comparison
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.plot(ghia_u_val, ghia_u_y, 'ro', label='Ghia et al. (1982)')
    plt.plot(u_centerline_pinn, y_test_coords.cpu().numpy(), 'b-', label='PINN Prediction')
    plt.title('U-velocity at X = 0.5 (Re=100)')
    plt.xlabel('U-velocity')
    plt.ylabel('Y-coordinate')
    plt.grid(True)
    plt.legend()
    plt.xlim([-0.3, 1.1]) # Adjusted xlim to better fit new Ghia U-velocity range
    plt.ylim([0, 1])

    plt.subplot(1, 2, 2)
    plt.plot(ghia_v_x, ghia_v_val, 'ro', label='Ghia et al. (1982)')
    plt.plot(x_test_coords.cpu().numpy(), v_centerline_pinn, 'b-', label='PINN Prediction')
    plt.title('V-velocity at Y = 0.5 (Re=100)')
    plt.xlabel('X-coordinate')
    plt.ylabel('V-velocity')
    plt.grid(True)
    plt.legend()
    plt.xlim([0, 1])
    plt.ylim([-0.3, 0.3]) # Adjusted ylim to better fit new Ghia V-velocity range

    plt.tight_layout()
    plt.show()